In [1]:
import numpy as np

# Function based on Rescale4DL

def find_matching_labels(gt: np.array, pred: np.array):
    """
    Find the matching labels between ground truth and prediction. If a pred_label
    has no assignment in the ground truth, assign it to 0.

    Parameters
    ----------
    gt : np.array
        Ground truth labels.
    pred : np.array
        Predicted labels.

    Returns
    -------
    list
        List of tuples (gt_label, pred_label, score).
    """
    if np.unique(pred).shape[0] == 1:
        return ((gt_lbl, 0, 0) for gt_lbl in np.unique(gt))

    scores = compute_labels_matching_scores(gt, pred)
    pred_labels = np.unique(pred)

    # Process scores to resolve conflicts and get final matching labels
    matching_labels = remove_duplicates(scores, pred_labels)
    return matching_labels

def compute_labels_matching_scores(gt: np.array, pred: np.array):
    """
    Compute matching scores between ground truth and predicted labels.

    Parameters
    ----------
    gt : np.array
        Ground truth labels.
    pred : np.array
        Predicted labels.

    Returns
    -------
    dict
        Dictionary with gt_label as keys and a list of tuples (pred_label, score) as values.
    """
    scores = {}
    gt_labels = np.unique(gt)

    for lbl in gt_labels[1:]:  # skips the background label
        scores[lbl] = []
        rows_idx, cols_idx = np.nonzero(gt == lbl)
        min_row, max_row, min_col, max_col = (
            np.min(rows_idx),
            np.max(rows_idx),
            np.min(cols_idx),
            np.max(cols_idx),
        )
        pred_box = pred[min_row : max_row + 1, min_col : max_col + 1]
        pred_labels_in_box = np.unique(pred_box)
        for pred_lbl in pred_labels_in_box:
            score = score_label_overlap(gt, pred, lbl, pred_lbl)
            scores[lbl].append([pred_lbl, score])

        scores[lbl] = sorted(scores[lbl], key=lambda x: x[1], reverse=True)

    return scores

def score_label_overlap(gt: np.array, pred: np.array, gt_label, pred_label):
    """
    Calculate the score of label overlap between ground truth and prediction.

    Parameters
    ----------
    gt : np.array
        Ground truth labels.
    pred : np.array
        Predicted labels.
    gt_label : int
        Label in ground truth.
    pred_label : int
        Label in prediction.

    Returns
    -------
    float
        Score of label overlap.
    """
    gt_mask = gt == gt_label
    pred_mask = pred == pred_label

    intersection = np.sum(gt_mask & pred_mask)
    union = np.sum(gt_mask | pred_mask)

    if union == 0:
        score = 0.0
    else:
        score = intersection / union

    return score

def remove_duplicates(scores, pred_labels):
    """
    Resolve conflicts in the scores dictionary by ensuring each pred_label
    is assigned to the gt_label with the highest score. If a pred_label has no
    assignment in the ground truth, assign it to 0.

    Parameters
    ----------
    scores : dict
        Dictionary with gt_label as keys and a list of tuples (pred_label, score) as values.
    pred_labels : np.array
        Array of unique predicted labels.

    Returns
    -------
    list
        List of tuples (gt_label, pred_label, score) with resolved conflicts.
    """
    assigned_pred_labels = set()
    result = []

    # Sort gt_labels by their highest score to prioritize them
    sorted_gt_labels = sorted(
        scores.keys(),
        key=lambda lbl: scores[lbl][0][1] if scores[lbl] else 0,
        reverse=True,
    )

    for gt_label in sorted_gt_labels:
        for pred_label, score in scores[gt_label]:
            if pred_label not in assigned_pred_labels:
                result.append((gt_label, pred_label, score))
                assigned_pred_labels.add(pred_label)
                break

    # Add unmatched pred_labels with gt_label = 0
    for pred_label in pred_labels:
        if pred_label not in assigned_pred_labels:
            result.append((0, pred_label, 0.0))

    return result


# SEGMENTATION STARDIST SAUREUS

In [2]:
import tifffile
import os
import numpy as np
import matplotlib.pyplot as plt

from stardist.models import StarDist2D
from napari_mAIcrobe.mAIcrobe.unet import computelabel_unet, normalizePercentile


rootfolder = "SegmentationStarDistSaureus/StarDistSaureus/TEST"
path2imgs_memb = [rootfolder + "/Images/wt"+str(i)+".tif" for i in [1,2,3]]
path2imgs_gt = [rootfolder + "/Labels/wt"+str(i)+".tif" for i in [1,2,3]]

imgs_memb = [tifffile.imread(path) for path in path2imgs_memb]
labels_gt = [tifffile.imread(path) for path in path2imgs_gt]
mask_gt = [gt>0 for gt in labels_gt]


def stardist(imgmemb):
    basedir,name = os.path.split("SegmentationStarDistSaureus/StarDistModel")
    model = StarDist2D(None, name = name, basedir= basedir) 

    labelsSD, _ = model.predict_instances(normalizePercentile(imgmemb))
    maskSD = labelsSD > 0
    maskSD = maskSD.astype('uint16')

    return maskSD,labelsSD


maskSDs = []
labelsSDs = []
for imgmemb in imgs_memb:
    maskSD, labelsSD = stardist(imgmemb)
    maskSDs.append(maskSD)
    labelsSDs.append(labelsSD)

matching_sd = [find_matching_labels(lgt, lsd) for lgt, lsd in zip(labels_gt, labelsSDs)]

# gt, sd, iou
print(matching_sd)


# total IOU's
iou_sd = [(mgt & msd).sum() / (mgt | msd).sum() for mgt, msd in zip(mask_gt, maskSDs)]
print(iou_sd)

# total IoU
total_iou_sd = sum([(mgt & msd).sum() for mgt, msd in zip(mask_gt, maskSDs)]) / sum([(mgt | msd).sum() for mgt, msd in zip(mask_gt, maskSDs)])
print(total_iou_sd)

# avg iou's per label 
avg_iou_sd = [np.average([ml[2] for ml in matching]) for matching in matching_sd]
print(avg_iou_sd)


tp = 0
fn = 0
fp = 0
for matchin in matching_sd:
    for m in matchin:
        if m[0] == 0 and m[1]!=0:
            fp += 1
        elif m[0] != 0 and m[1]==0:
            fn += 1
        elif m[0] != 0 and m[1] != 0:
            tp += 1
 


recall = tp / (tp+fn)
precision = tp / (tp+fp)

print("Recall: ", recall)
print("Precision: ", precision)

print(iou_sd, np.average(iou_sd))
 

Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.479645, nms_thresh=0.3.
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.479645, nms_thresh=0.3.
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.479645, nms_thresh=0.3.
[[(103, 346, 0.9705488621151271), (238, 177, 0.9698660714285714), (507, 29, 0.9695756605284227), (179, 122, 0.96875), (106, 176, 0.9678819444444444), (185, 87, 0.9672943508424182), (218, 272, 0.966796875), (154, 320, 0.9664948453608248), (213, 44, 0.9661157024793389), (80, 332, 0.9657387580299786), (8, 16, 0.9636711281070746), (150, 55, 0.9616390584132519), (236, 136, 0.9613259668508287), (116, 113, 0.9607623318385651), (63, 92, 0.96), (182, 63, 0.9598633646456021), (170, 262, 0.9593253968253969), (109, 378, 0.9576185671039354), (117, 135, 0

# STREP UNET

In [3]:
from skimage.io import imread as imreadskimage

rootfolder = "SegmentationStrepUnet/Test"
path2imgs_memb = [rootfolder + "/Phase/"+str(i)+".tif" for i in [1,2,3]]
path2imgs_gt = [rootfolder + "/Labels/"+str(i)+".png" for i in [1,2,3]]

imgs_memb = [tifffile.imread(path) for path in path2imgs_memb]
labels_gt = [imreadskimage(path) for path in path2imgs_gt]
mask_gt = [gt>0 for gt in labels_gt]

def unet(imgmemb):
    maskU, labelsU = computelabel_unet(path2model="SegmentationStrepUnet/STREPPHASE_2025may19/modelpath/phasemodel_20250519/weights_best.hdf5", base_image=imgmemb, closing=0, dilation=0, fillholes=False)

    return maskU, labelsU

maskUs = []
labelsUs = []
for imgmemb in imgs_memb:
    maskU, labelsU = unet(imgmemb)
    maskUs.append(maskU)
    labelsUs.append(labelsU)
matching_unet = [find_matching_labels(lgt, lunet) for lgt, lunet in zip(labels_gt, labelsUs)]


# total IOU's
iou_u = [(mgt & msd).sum() / (mgt | msd).sum() for mgt, msd in zip(mask_gt, maskUs)]
print(iou_u)

# total IoU
total_iou_u = sum([(mgt & msd).sum() for mgt, msd in zip(mask_gt, maskUs)]) / sum([(mgt | msd).sum() for mgt, msd in zip(mask_gt, maskUs)])
print(total_iou_u)

# avg iou's per label 
avg_iou_u = [np.average([ml[2] for ml in matching]) for matching in matching_unet]
print(avg_iou_u)


tp = 0
tn = 0
fp = 0
for matchin in matching_unet:
    for m in matchin:
        if m[0] == 0 and m[1]!=0:
            fp += 1
        elif m[0] != 0 and m[1]==0:
            fn += 1
        elif m[0] != 0 and m[1] != 0:
            tp += 1
 


recall = tp / (tp+fn)
precision = tp / (tp+fp)

print("Recall: ", recall)
print("Precision: ", precision)

 
 
print(iou_u, np.average(iou_u))

1/1 [==============================] - 0s 189ms/step


1/1 [==============================] - 0s 204ms/step


1/1 [==============================] - 0s 203ms/step
[0.892663216287875, 0.9009281778924902, 0.8992682843821257]
0.8972984643111929
[0.8548281683843704, 0.872589028466741, 0.855799687809663]
Recall:  0.9976700838769804
Precision:  0.9875461254612546
[0.892663216287875, 0.9009281778924902, 0.8992682843821257] 0.8976198928541637


# BACILLUS UNET

In [4]:
def unet(imgmemb):
    maskU, labelsU = computelabel_unet(path2model="SegmentationUNetBacillus/UNET4DeepbacsBacillus/modelpath/FINETUNED_staph2bacillus/weights_best.hdf5", base_image=imgmemb, closing=0, dilation=0, fillholes=False)

    return maskU, labelsU

rootfolder = "SegmentationUNetBacillus/Test"

path2imgs_memb = [rootfolder + "/fluorescence/test_"+str(i)+".tif" for i in [1,2,3]]
path2imgs_gt = [rootfolder + "/masks/test_"+str(i)+".tif" for i in [1,2,3]]

imgs_memb = [tifffile.imread(path) for path in path2imgs_memb]
labels_gt = [tifffile.imread(path) for path in path2imgs_gt]
mask_gt = [gt>0 for gt in labels_gt]

maskUs = []
labelsUs = []
for imgmemb in imgs_memb:
    maskU, labelsU = unet(imgmemb)
    maskUs.append(maskU)
    labelsUs.append(labelsU)
matching_unet_bs = [find_matching_labels(lgt, lunet) for lgt, lunet in zip(labels_gt, labelsUs)]

    
# total IOU's
iou_u = [(mgt & msd).sum() / (mgt | msd).sum() for mgt, msd in zip(mask_gt, maskUs)]
print(iou_u, np.average(iou_u))

tp = 0
tn = 0
fp = 0
for matchin in matching_unet_bs:
    for m in matchin:
        if m[0] == 0 and m[1]!=0:
            fp += 1
        elif m[0] != 0 and m[1]==0:
            fn += 1
        elif m[0] != 0 and m[1] != 0:
            tp += 1
 


recall = tp / (tp+fn)
precision = tp / (tp+fp)

print("Recall: ", recall)
print("Precision: ", precision)

 
 
print(iou_u, np.average(iou_u))

1/1 [==============================] - 1s 775ms/step


1/1 [==============================] - 1s 718ms/step


1/1 [==============================] - 1s 785ms/step
[0.8436221731158023, 0.8445307671934317, 0.8319878506913916] 0.8400469303335418
Recall:  0.9230769230769231
Precision:  1.0
[0.8436221731158023, 0.8445307671934317, 0.8319878506913916] 0.8400469303335418
